<a href="https://colab.research.google.com/github/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification/blob/main/wk9_llm_synthetic_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install datasets openai scikit-learn -q

In [4]:
import pandas as pd
import numpy as np
import time
import json
import re
from datasets import load_dataset
from openai import OpenAI
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report

# ── GLOBAL CONSTANTS ─────────────────────────────────────────────────────────
# All parameters defined here — never hardcoded elsewhere in the notebook

RANDOM_STATE = 42
TEST_PARTICIPANTS = ["participant_14", "participant_16", "participant_20"]

# Features used for generation and classification
# Excludes: zero-variance (Air-Velocity=0.1, Metabolic-Rate=1.0)
# Excludes: pose keypoints, emotion columns, file metadata, timestamp
# Excludes: participant_id (derived column, not a sensor feature)
FEATURE_COLS = [
    "Age", "Gender", "Weight", "Height", "Bodyfat", "Bodytemp",
    "Sport-Last-Hour", "Time-Since-Meal", "Tiredness", "Clothing-Level",
    "Radiation-Temp", "PCE-Ambient-Temp", "Wrist_Skin_Temperature",
    "Heart_Rate", "GSR", "Ambient_Temperature",
    "Ambient_Humidity", "Solar_Radiation"
]

# Columns to drop from feature matrix — explicitly defined, never mutated
# BUG-2 prevention: target columns explicitly listed here
DROP_COLS = [
    "file_name", "Timestamp", "participant_id",
    "Label",           # 7-class target
    "Label_3class",    # 3-class target — added when used
    "Air-Velocity",    # zero variance
    "Metabolic-Rate",  # zero variance
    "Nose", "Neck", "RShoulder", "RElbow", "LShoulder",
    "LElbow", "REye", "LEye", "REar", "LEar",
    "Emotion-Self", "Emotion-ML",
]

TARGET_COL_7 = "Label"
TARGET_COL_3 = "Label_3class"

print("Constants defined.")
print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Test participants: {TEST_PARTICIPANTS}")

Constants defined.
Feature columns: 18
Test participants: ['participant_14', 'participant_16', 'participant_20']


In [5]:
# ── DATA LOADING ──────────────────────────────────────────────────────────────
# Uses subject-wise split consistent with all previous experiments in this project
# Random row-wise splits are deliberately avoided — AutoTherm samples at 42Hz
# so consecutive rows from the same participant are highly correlated.
# A random split would cause within-subject leakage (METH-3).

dataset = load_dataset("kopetri/AutoTherm", "indoor")
df = dataset["train"].to_pandas()

def extract_participant_id(filename):
    """Extract participant ID from AutoTherm filename."""
    match = re.search(r"participant_\d+", filename)
    return match.group() if match else "unknown"

df["participant_id"] = df["file_name"].apply(extract_participant_id)

# Subject-wise split
train_df = df[~df["participant_id"].isin(TEST_PARTICIPANTS)].copy()
test_df  = df[df["participant_id"].isin(TEST_PARTICIPANTS)].copy()

print(f"Train participants: {sorted(train_df['participant_id'].unique())}")
print(f"Test participants:  {sorted(test_df['participant_id'].unique())}")
print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"\nCold (-3) in training set: {(train_df['Label']==-3).sum()} rows "
      f"({(train_df['Label']==-3).mean()*100:.2f}%)")
print(f"Cold (-3) in test set:     {(test_df['Label']==-3).sum()} rows "
      f"({(test_df['Label']==-3).mean()*100:.2f}%)")

Train participants: ['participant_10', 'participant_11', 'participant_15', 'participant_17', 'participant_18', 'participant_19', 'participant_2', 'participant_21', 'participant_3', 'participant_4', 'participant_6', 'participant_7', 'participant_8']
Test participants:  ['participant_14', 'participant_16', 'participant_20']
Train shape: (1276709, 36)
Test shape:  (290019, 36)

Cold (-3) in training set: 48020 rows (3.76%)
Cold (-3) in test set:     22556 rows (7.78%)


In [7]:
# ── SHARED UTILITY FUNCTIONS ──────────────────────────────────────────────────
# BUG-1 FIX: LabelEncoder fitted ONCE on training data only.
# Never refitted on test data. Consistent with wk8_shared_utils.ipynb.

def add_3class_label(df):
    """Add collapsed 3-class label. Cold≤-2=0, Neutral-1to+1=1, Warm≥+2=2."""
    df = df.copy()
    df[TARGET_COL_3] = df[TARGET_COL_7].apply(
        lambda x: 0 if x <= -2 else (2 if x >= 2 else 1)
    )
    return df

def fit_encoders(train_df):
    """Fit LabelEncoders on training data only. Returns dict of fitted encoders.

    BUG-1 FIX: encoders fitted once here, applied to both train and test
    via apply_encoders(). Never call fit() on test data.
    """
    X = train_df.drop(
        columns=[c for c in DROP_COLS if c in train_df.columns]
    )
    encoders = {}
    for col in X.select_dtypes(include=["object", "category"]).columns:
        le = LabelEncoder()
        le.fit(X[col].astype(str))
        encoders[col] = le
    return encoders

def prepare_features(df, target_col, encoders):
    """Prepare feature matrix and target vector.

    BUG-1 FIX: applies pre-fitted encoders, never refits.
    BUG-2 FIX: drops target_col explicitly via DROP_COLS constant.
    Unseen categories default to -1 to avoid crashes on test data.

    Args:
        df: DataFrame for this split
        target_col: name of target column to predict
        encoders: dict of fitted LabelEncoders from fit_encoders()

    Returns:
        X (numpy array), y (numpy array)
    """
    df = df.copy()
    y = df[target_col].values

    # Build drop list — always includes target_col explicitly
    # BUG-2 prevention: target never survives into feature matrix
    drop = list(set(DROP_COLS + [target_col]))
    X = df.drop(columns=[c for c in drop if c in df.columns])

    # Apply pre-fitted encoders only — never refit
    for col, le in encoders.items():
        if col in X.columns:
            X[col] = X[col].astype(str).map(
                lambda val, le=le: (
                    le.transform([val])[0]
                    if val in le.classes_ else -1
                )
            )

    X = X.fillna(X.median(numeric_only=True))
    return X.values, y

# Add 3-class labels to both splits
train_df = add_3class_label(train_df)
test_df  = add_3class_label(test_df)

# Fit encoders on training data only
encoders = fit_encoders(train_df)
print(f"Encoders fitted: {list(encoders.keys())}")

# Prepare real baseline features
X_train_real, y_train_real = prepare_features(train_df, TARGET_COL_7, encoders)
X_test, y_test             = prepare_features(test_df,  TARGET_COL_7, encoders)

X_train_real_3, y_train_real_3 = prepare_features(train_df, TARGET_COL_3, encoders)
X_test_3, y_test_3             = prepare_features(test_df,  TARGET_COL_3, encoders)

# Verify no leakage — feature count should be 18
print(f"\nFeature matrix shape (7-class): X_train={X_train_real.shape}, X_test={X_test.shape}")
print(f"Feature matrix shape (3-class): X_train={X_train_real_3.shape}, X_test={X_test_3.shape}")
print(f"\nBUG-2 check — feature count: {X_train_real.shape[1]} (expected 18)")

Encoders fitted: ['Gender']

Feature matrix shape (7-class): X_train=(1276709, 18), X_test=(290019, 18)
Feature matrix shape (3-class): X_train=(1276709, 18), X_test=(290019, 18)

BUG-2 check — feature count: 18 (expected 18)


In [8]:
# ── REAL DATA BASELINE ────────────────────────────────────────────────────────
# Establishes the benchmark all synthetic methods are compared against.
# Latency tracked as Mark requested — report inference time alongside F1.

print("Training baseline Random Forest on real data...")
start_train = time.time()

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_baseline.fit(X_train_real, y_train_real)
train_time = time.time() - start_train

# Inference with latency
start_infer = time.time()
y_pred_baseline = rf_baseline.predict(X_test)
infer_time = time.time() - start_infer
latency_per_row = (infer_time / len(X_test)) * 1000  # ms per row

baseline_f1_7 = f1_score(y_test, y_pred_baseline,
                          average="macro", zero_division=0)
baseline_cold_f1 = f1_score(y_test, y_pred_baseline,
                              average=None,
                              labels=[-3,-2,-1,0,1,2,3],
                              zero_division=0)[0]

# 3-class baseline
rf_baseline_3 = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_baseline_3.fit(X_train_real_3, y_train_real_3)
y_pred_baseline_3 = rf_baseline_3.predict(X_test_3)
baseline_f1_3 = f1_score(y_test_3, y_pred_baseline_3,
                           average="macro", zero_division=0)

print(f"\nBaseline Results:")
print(f"  7-class Macro F1:  {baseline_f1_7:.4f}")
print(f"  3-class Macro F1:  {baseline_f1_3:.4f}")
print(f"  Cold F1:           {baseline_cold_f1:.4f}")
print(f"  Train time:        {train_time:.1f}s")
print(f"  Inference latency: {latency_per_row:.4f} ms/row")
print(f"  Total inference:   {infer_time:.2f}s for {len(X_test):,} rows")

Training baseline Random Forest on real data...

Baseline Results:
  7-class Macro F1:  0.2858
  3-class Macro F1:  0.7163
  Cold F1:           0.0000
  Train time:        75.3s
  Inference latency: 0.0031 ms/row
  Total inference:   0.91s for 290,019 rows


In [9]:
# ── LLM SYNTHETIC DATA GENERATION SETUP ──────────────────────────────────────
# Uses GPT-4o-mini for generation (temperature=0 for reproducibility).
# Goal: generate synthetic Cold (-3) sensor rows using the LLM's general
# knowledge of human physiology, augmented with real Cold examples as context.
#
# This is distinct from LLM classification (wk8): here the LLM acts as a
# DATA GENERATOR, not a classifier. Generated rows are then used to train
# a downstream Random Forest classifier (TSTR and augmented protocols).

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# Get real Cold examples to show the LLM what Cold looks like in this dataset
# Sample from training set only — never from test set
cold_real = train_df[train_df["Label"] == -3][FEATURE_COLS].sample(
    5, random_state=RANDOM_STATE
)

# Get feature statistics from training data for grounding the generation
feature_stats = train_df[FEATURE_COLS].describe()

print("Real Cold examples (shown to LLM as context):")
print(cold_real.to_string())
print(f"\nFeature statistics computed from training data.")

# Verify API connection
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say: ready"}],
    max_tokens=5,
    temperature=0
)
print(f"\nAPI status: {resp.choices[0].message.content}")

Real Cold examples (shown to LLM as context):
         Age  Gender  Weight  Height  Bodyfat  Bodytemp  Sport-Last-Hour  Time-Since-Meal  Tiredness  Clothing-Level  Radiation-Temp  PCE-Ambient-Temp  Wrist_Skin_Temperature  Heart_Rate       GSR  Ambient_Temperature  Ambient_Humidity  Solar_Radiation
768335    20  Female    57.6   170.0    0.246      36.5                0               16          3            0.57            23.6              24.1                   34.63       51.20  0.457620                 23.6              41.0         0.257136
1066998   25  Female    54.2   155.0    0.000      36.5                0                5          3            0.61            23.6              23.3                   34.01       78.36  0.380754                 22.6              33.0         0.251232
1068100   25  Female    54.2   155.0    0.000      36.5                0                5          3            0.61            23.3              23.1                   33.87       81.70  0.37306

In [10]:
# ── LLM GENERATION PROMPT ─────────────────────────────────────────────────────
# Strategy: show the LLM real Cold examples as context, then ask it to generate
# new Cold rows with variation. Each generated row must have all 18 features.
# Temperature=0 for reproducibility.

def build_generation_prompt(cold_examples_df, n_to_generate=10):
    """Build a prompt asking GPT to generate synthetic Cold sensor rows.

    Shows real Cold examples as context so the model understands the
    feature ranges and physiological patterns associated with Cold sensation.
    Asks for JSON output for reliable parsing.

    Args:
        cold_examples_df: DataFrame of real Cold examples to show as context
        n_to_generate: number of synthetic rows to request per call

    Returns:
        prompt string
    """
    # Format real examples as readable context
    examples_text = ""
    for i, (_, row) in enumerate(cold_examples_df.iterrows()):
        examples_text += f"\nExample {i+1} (Label: Cold, -3):\n"
        for feat in FEATURE_COLS:
            examples_text += f"  {feat}: {row[feat]}\n"

    feature_list = ", ".join(FEATURE_COLS)

    prompt = f"""You are a physiological sensor data expert. Your task is to generate
realistic synthetic wearable sensor readings for people experiencing COLD thermal
discomfort (label = -3 on the ASHRAE 7-point thermal sensation scale from -3 Cold to +3 Hot).

Here are {len(cold_examples_df)} real examples of Cold sensation sensor readings
from the AutoTherm indoor dataset:
{examples_text}

Generate {n_to_generate} NEW synthetic sensor rows for people experiencing Cold
sensation. Each row must:
1. Be physiologically plausible — values must fall within realistic human ranges
2. Show variation — do not simply copy the examples above
3. Reflect Cold conditions — lower ambient temperatures, lower wrist skin temperatures
4. Include exactly these {len(FEATURE_COLS)} features: {feature_list}

Important constraints from the real data:
- Age: typically 20-35 years
- Gender: Male or Female
- Ambient_Temperature: typically 18-24°C for Cold sensation
- Wrist_Skin_Temperature: typically 31-34°C for Cold sensation
- Clothing_Level: 0.4-0.8 (light to moderate clothing)
- GSR: 0.0-2.0
- Heart_Rate: 45-100 bpm

Respond with ONLY a valid JSON array of {n_to_generate} objects.
Each object must have exactly these keys: {feature_list}
No explanation, no markdown, no code blocks — just the raw JSON array."""

    return prompt


# Test the prompt on first row
test_prompt = build_generation_prompt(cold_real, n_to_generate=3)
print("Prompt built successfully.")
print(f"Prompt length: {len(test_prompt)} characters")
print("\nFirst 500 characters:")
print(test_prompt[:500])

Prompt built successfully.
Prompt length: 3697 characters

First 500 characters:
You are a physiological sensor data expert. Your task is to generate 
realistic synthetic wearable sensor readings for people experiencing COLD thermal 
discomfort (label = -3 on the ASHRAE 7-point thermal sensation scale from -3 Cold to +3 Hot).

Here are 5 real examples of Cold sensation sensor readings 
from the AutoTherm indoor dataset:

Example 1 (Label: Cold, -3):
  Age: 20
  Gender: Female
  Weight: 57.6
  Height: 170.0
  Bodyfat: 0.246
  Bodytemp: 36.5
  Sport-Last-Hour: 0
  Time-Since-M


In [12]:
# ── GENERATION FUNCTION ───────────────────────────────────────────────────────

def generate_synthetic_cold_rows(n_total=100, batch_size=10, model="gpt-4o-mini"):
    """Generate synthetic Cold sensor rows using LLM.

    Generates in batches to manage API rate limits and token limits.
    Records latency per batch for reporting as Mark requested.
    Validates each generated row has all required features.

    Args:
        n_total: total number of synthetic Cold rows to generate
        batch_size: rows per API call
        model: OpenAI model to use

    Returns:
        DataFrame of synthetic Cold rows with Label=-3
        dict of latency statistics
    """
    all_rows = []
    latencies = []
    errors = 0
    n_batches = n_total // batch_size

    print(f"Generating {n_total} synthetic Cold rows in {n_batches} batches "
          f"of {batch_size}...")

    for batch_idx in range(n_batches):
        prompt = build_generation_prompt(cold_real, n_to_generate=batch_size)

        start = time.time()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=3000,
                temperature=0
            )
            elapsed = time.time() - start
            latencies.append(elapsed)

            # Parse JSON response
            content = response.choices[0].message.content.strip()
            # Remove markdown code blocks if present
            content = content.replace("```json", "").replace("```", "").strip()

            batch_rows = json.loads(content)

            # Validate each row has all required features
            valid_rows = []
            for row in batch_rows:
                if all(f in row for f in FEATURE_COLS):
                    valid_rows.append(row)
                else:
                    missing = [f for f in FEATURE_COLS if f not in row]
                    print(f"  Batch {batch_idx+1}: row missing features: {missing}")

            all_rows.extend(valid_rows)

        except json.JSONDecodeError as e:
            print(f"  Batch {batch_idx+1}: JSON parse error — {e}")
            errors += 1
        except Exception as e:
            print(f"  Batch {batch_idx+1}: API error — {e}")
            errors += 1

        time.sleep(0.5)  # Rate limit buffer

        if (batch_idx + 1) % 2 == 0:
            print(f"  Progress: {batch_idx+1}/{n_batches} batches, "
                  f"{len(all_rows)} valid rows so far")

    # Build DataFrame
    synthetic_df = pd.DataFrame(all_rows)[FEATURE_COLS]
    synthetic_df["Label"] = -3  # All generated rows are Cold

    # Latency statistics
    latency_stats = {
        "mean_latency_s": np.mean(latencies),
        "std_latency_s": np.std(latencies),
        "total_time_s": sum(latencies),
        "rows_generated": len(synthetic_df),
        "errors": errors,
        "latency_per_row_ms": (np.mean(latencies) / batch_size) * 1000
    }

    print(f"\nGeneration complete:")
    print(f"  Valid rows generated: {len(synthetic_df)}")
    print(f"  Errors: {errors}")
    print(f"  Mean latency per batch: {latency_stats['mean_latency_s']:.2f}s")
    print(f"  Latency per row: {latency_stats['latency_per_row_ms']:.2f}ms")
    print(f"  Total generation time: {latency_stats['total_time_s']:.1f}s")

    return synthetic_df, latency_stats


# Test with 1 batch first to verify output quality
print("Running test batch (10 rows)...")
test_df_gen, test_stats = generate_synthetic_cold_rows(
    n_total=10, batch_size=10
)
print(f"\nTest batch output:")
print(test_df_gen.head(3))
print(f"\nData types:\n{test_df_gen.dtypes}")

Running test batch (10 rows)...
Generating 10 synthetic Cold rows in 1 batches of 10...

Generation complete:
  Valid rows generated: 10
  Errors: 0
  Mean latency per batch: 23.60s
  Latency per row: 2360.40ms
  Total generation time: 23.6s

Test batch output:
   Age  Gender  Weight  Height  Bodyfat  Bodytemp  Sport-Last-Hour  \
0   22  Female    50.0   160.0     0.22      36.4                0   
1   30    Male    85.0   180.0     0.18      36.3                0   
2   28  Female    62.0   165.0     0.25      36.2                0   

   Time-Since-Meal  Tiredness  Clothing-Level  Radiation-Temp  \
0                4          2            0.65            21.5   
1                2          4            0.55            19.8   
2                6          3            0.70            20.0   

   PCE-Ambient-Temp  Wrist_Skin_Temperature  Heart_Rate    GSR  \
0              21.0                    33.2        72.5  0.512   
1              19.5                    32.5        68.0  0.800  

In [13]:
# ── FULL GENERATION RUN ───────────────────────────────────────────────────────
# Generate 100 synthetic Cold rows — sufficient for augmentation experiment
# while keeping API cost manageable (~$0.50 estimated)

print("Generating full synthetic Cold dataset (100 rows)...")
synthetic_cold_df, generation_stats = generate_synthetic_cold_rows(
    n_total=100,
    batch_size=10,
    model="gpt-4o-mini"
)

print(f"\nGenerated dataset shape: {synthetic_cold_df.shape}")
print(f"Label distribution: {synthetic_cold_df['Label'].value_counts().to_dict()}")
print(f"\nGeneration statistics:")
for k, v in generation_stats.items():
    print(f"  {k}: {v}")

# Save generated data
synthetic_cold_df.to_csv("llm_synthetic_cold_rows.csv", index=False)
print("\nSaved to llm_synthetic_cold_rows.csv")

Generating full synthetic Cold dataset (100 rows)...
Generating 100 synthetic Cold rows in 10 batches of 10...
  Progress: 2/10 batches, 20 valid rows so far
  Progress: 4/10 batches, 40 valid rows so far
  Progress: 6/10 batches, 60 valid rows so far
  Progress: 8/10 batches, 80 valid rows so far
  Progress: 10/10 batches, 100 valid rows so far

Generation complete:
  Valid rows generated: 100
  Errors: 0
  Mean latency per batch: 21.42s
  Latency per row: 2142.21ms
  Total generation time: 214.2s

Generated dataset shape: (100, 19)
Label distribution: {-3: 100}

Generation statistics:
  mean_latency_s: 21.422141218185423
  std_latency_s: 3.189188806428441
  total_time_s: 214.22141218185425
  rows_generated: 100
  errors: 0
  latency_per_row_ms: 2142.2141218185425

Saved to llm_synthetic_cold_rows.csv


In [14]:
# ── FIDELITY CHECK ────────────────────────────────────────────────────────────
# Before classification evaluation, check whether generated rows are
# statistically plausible compared to real Cold rows.

real_cold = train_df[train_df["Label"] == -3][FEATURE_COLS].copy()
synth_cold = synthetic_cold_df[FEATURE_COLS].copy()

# Encode Gender for comparison
synth_cold["Gender"] = synth_cold["Gender"].map(
    lambda v: encoders["Gender"].transform([str(v)])[0]
    if str(v) in encoders["Gender"].classes_ else -1
)

print("Feature comparison — Real Cold vs Synthetic Cold:")
print(f"{'Feature':<30} {'Real Mean':>12} {'Synth Mean':>12} {'Real Std':>10} {'Synth Std':>10}")
print("-" * 78)

numeric_features = [f for f in FEATURE_COLS if f != "Gender"]
for feat in numeric_features:
    try:
        real_mean = real_cold[feat].mean()
        synth_mean = synth_cold[feat].mean()
        real_std = real_cold[feat].std()
        synth_std = synth_cold[feat].std()
        print(f"{feat:<30} {real_mean:>12.3f} {synth_mean:>12.3f} "
              f"{real_std:>10.3f} {synth_std:>10.3f}")
    except:
        pass

Feature comparison — Real Cold vs Synthetic Cold:
Feature                           Real Mean   Synth Mean   Real Std  Synth Std
------------------------------------------------------------------------------
Age                                  23.294       26.100      2.371      3.317
Weight                               64.707       70.340     17.539     15.931
Height                              167.756      171.460     13.804     10.468
Bodyfat                               0.084        0.199      0.117      0.058
Bodytemp                             36.373       36.212      0.245      0.147
Sport-Last-Hour                       0.000        0.030      0.000      0.171
Time-Since-Meal                       8.331        3.930      5.571      1.810
Tiredness                             3.633        3.450      1.224      1.048
Clothing-Level                        0.588        0.617      0.020      0.077
Radiation-Temp                       21.957       20.421      1.619      0.950
PC

In [15]:
# ── TSTR EVALUATION ───────────────────────────────────────────────────────────
# Train Synthetic Test Real: train classifier ONLY on synthetic Cold rows
# combined with real non-Cold rows, then evaluate on real test set.
#
# Protocol: replace real Cold training rows with LLM-generated Cold rows.
# This directly tests whether LLM-generated Cold data can substitute for
# real Cold data in training.

print("Preparing TSTR training set...")

# Real training data without Cold rows
train_no_cold = train_df[train_df["Label"] != -3].copy()

# Add 3-class label to synthetic data
synthetic_cold_df_labeled = synthetic_cold_df.copy()
synthetic_cold_df_labeled[TARGET_COL_3] = 0  # Cold = 0 in 3-class

# Combine: real non-Cold + synthetic Cold
train_tstr = pd.concat([train_no_cold, synthetic_cold_df_labeled],
                        ignore_index=True)

print(f"TSTR training set shape: {train_tstr.shape}")
print(f"Label distribution in TSTR set:")
print(train_tstr["Label"].value_counts().sort_index())

# Prepare features for TSTR
# BUG-1: use same encoders fitted on original training data
X_train_tstr, y_train_tstr = prepare_features(
    train_tstr, TARGET_COL_7, encoders
)

print(f"\nX_train_tstr shape: {X_train_tstr.shape}")
print(f"BUG-2 check: {X_train_tstr.shape[1]} features (expected 18)")

# Train TSTR classifier
print("\nTraining TSTR Random Forest...")
start = time.time()
rf_tstr = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_tstr.fit(X_train_tstr, y_train_tstr)
tstr_train_time = time.time() - start

# Evaluate on real test set
start = time.time()
y_pred_tstr = rf_tstr.predict(X_test)
tstr_infer_time = time.time() - start
tstr_latency = (tstr_infer_time / len(X_test)) * 1000

tstr_f1_7 = f1_score(y_test, y_pred_tstr,
                      average="macro", zero_division=0)
tstr_cold_f1 = f1_score(y_test, y_pred_tstr,
                          average=None,
                          labels=[-3,-2,-1,0,1,2,3],
                          zero_division=0)[0]

print(f"\nTSTR Results (LLM synthetic Cold replacing real Cold):")
print(f"  7-class Macro F1:  {tstr_f1_7:.4f}  (baseline: {baseline_f1_7:.4f})")
print(f"  Cold F1:           {tstr_cold_f1:.4f}")
print(f"  Train time:        {tstr_train_time:.1f}s")
print(f"  Inference latency: {tstr_latency:.4f} ms/row")
print(f"\nChange vs baseline: {tstr_f1_7 - baseline_f1_7:+.4f}")

Preparing TSTR training set...
TSTR training set shape: (1228789, 37)
Label distribution in TSTR set:
Label
-3       100
-2    116840
-1    263290
 0    307137
 1    178333
 2    202958
 3    160131
Name: count, dtype: int64

X_train_tstr shape: (1228789, 18)
BUG-2 check: 18 features (expected 18)

Training TSTR Random Forest...

TSTR Results (LLM synthetic Cold replacing real Cold):
  7-class Macro F1:  0.3032  (baseline: 0.2858)
  Cold F1:           0.0000
  Train time:        74.7s
  Inference latency: 0.0021 ms/row

Change vs baseline: +0.0174


In [16]:
# ── AUGMENTED TRAINING EVALUATION ─────────────────────────────────────────────
# Augmented: train on real data PLUS LLM synthetic Cold rows.
# Tests whether LLM-generated Cold rows add signal beyond real data alone.

print("Preparing augmented training set...")

# Add 3-class label to synthetic data
synth_for_aug = synthetic_cold_df.copy()
synth_for_aug[TARGET_COL_3] = 0  # Cold = 0 in 3-class

# Combine: full real training + synthetic Cold
train_aug = pd.concat([train_df, synth_for_aug], ignore_index=True)

print(f"Augmented training set shape: {train_aug.shape}")
print(f"Cold rows: real={48020}, synthetic={len(synth_for_aug)}, "
      f"total={(train_aug['Label']==-3).sum()}")

# Prepare features
X_train_aug, y_train_aug = prepare_features(
    train_aug, TARGET_COL_7, encoders
)

print(f"\nBUG-2 check: {X_train_aug.shape[1]} features (expected 18)")

# Train augmented classifier
print("\nTraining augmented Random Forest...")
start = time.time()
rf_aug = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_aug.fit(X_train_aug, y_train_aug)
aug_train_time = time.time() - start

# Evaluate
start = time.time()
y_pred_aug = rf_aug.predict(X_test)
aug_infer_time = time.time() - start
aug_latency = (aug_infer_time / len(X_test)) * 1000

aug_f1_7 = f1_score(y_test, y_pred_aug,
                     average="macro", zero_division=0)
aug_cold_f1 = f1_score(y_test, y_pred_aug,
                        average=None,
                        labels=[-3,-2,-1,0,1,2,3],
                        zero_division=0)[0]

print(f"\nAugmented Training Results:")
print(f"  7-class Macro F1:  {aug_f1_7:.4f}  (baseline: {baseline_f1_7:.4f})")
print(f"  Cold F1:           {aug_cold_f1:.4f}")
print(f"  Train time:        {aug_train_time:.1f}s")
print(f"  Inference latency: {aug_latency:.4f} ms/row")
print(f"\nChange vs baseline: {aug_f1_7 - baseline_f1_7:+.4f}")

Preparing augmented training set...
Augmented training set shape: (1276809, 37)
Cold rows: real=48020, synthetic=100, total=48120

BUG-2 check: 18 features (expected 18)

Training augmented Random Forest...

Augmented Training Results:
  7-class Macro F1:  0.2728  (baseline: 0.2858)
  Cold F1:           0.0000
  Train time:        76.2s
  Inference latency: 0.0021 ms/row

Change vs baseline: -0.0130


In [17]:
# ── FINAL RESULTS SUMMARY ─────────────────────────────────────────────────────

print("=" * 70)
print("LLM SYNTHETIC DATA GENERATION — COMPLETE RESULTS")
print("=" * 70)
print(f"\n{'Method':<40} {'7-class F1':>10} {'Cold F1':>10} {'vs Baseline':>12}")
print("-" * 70)
print(f"{'Baseline (real data only)':<40} {baseline_f1_7:>10.4f} "
      f"{0.00:>10.2f} {'—':>12}")
print(f"{'LLM TSTR (synth Cold replaces real)':<40} {tstr_f1_7:>10.4f} "
      f"{tstr_cold_f1:>10.2f} {tstr_f1_7-baseline_f1_7:>+12.4f}")
print(f"{'LLM Augmented (synth Cold added)':<40} {aug_f1_7:>10.4f} "
      f"{aug_cold_f1:>10.2f} {aug_f1_7-baseline_f1_7:>+12.4f}")
print("=" * 70)

print(f"\nGeneration Latency:")
print(f"  LLM generation:       {generation_stats['latency_per_row_ms']:.0f} ms/row")
print(f"  RF inference:         {tstr_latency:.4f} ms/row")
print(f"  Generation overhead:  {generation_stats['latency_per_row_ms']/tstr_latency:.0f}x slower than RF inference")

print(f"\nFidelity Issues Identified:")
print(f"  Ambient temp — Real Cold: {train_df[train_df['Label']==-3]['Ambient_Temperature'].mean():.2f}°C  "
      f"Synth: {synthetic_cold_df['Ambient_Temperature'].mean():.2f}°C")
print(f"  Wrist skin  — Real Cold: {train_df[train_df['Label']==-3]['Wrist_Skin_Temperature'].mean():.2f}°C  "
      f"Synth: {synthetic_cold_df['Wrist_Skin_Temperature'].mean():.2f}°C")

print(f"\nKey Findings:")
print(f"  1. LLM TSTR beats baseline (+0.0174) — synthetic Cold adds structural diversity")
print(f"  2. LLM Augmented underperforms baseline (-0.0130) — synthetic Cold introduces noise")
print(f"  3. Cold F1 = 0.00 under both protocols — fidelity mismatch prevents Cold learning")
print(f"  4. LLM overcools: generates Ambient_Temp ~19°C vs real Cold at ~22°C")
print(f"  5. Generation latency 2142ms/row vs RF inference 0.002ms/row — {generation_stats['latency_per_row_ms']/tstr_latency:.0f}x overhead")
print(f"  6. Zero generation errors across 100 rows — GPT-4o-mini follows JSON spec reliably")

LLM SYNTHETIC DATA GENERATION — COMPLETE RESULTS

Method                                   7-class F1    Cold F1  vs Baseline
----------------------------------------------------------------------
Baseline (real data only)                    0.2858       0.00            —
LLM TSTR (synth Cold replaces real)          0.3032       0.00      +0.0174
LLM Augmented (synth Cold added)             0.2728       0.00      -0.0130

Generation Latency:
  LLM generation:       2142 ms/row
  RF inference:         0.0021 ms/row
  Generation overhead:  999026x slower than RF inference

Fidelity Issues Identified:
  Ambient temp — Real Cold: 21.98°C  Synth: 19.08°C
  Wrist skin  — Real Cold: 33.39°C  Synth: 32.26°C

Key Findings:
  1. LLM TSTR beats baseline (+0.0174) — synthetic Cold adds structural diversity
  2. LLM Augmented underperforms baseline (-0.0130) — synthetic Cold introduces noise
  3. Cold F1 = 0.00 under both protocols — fidelity mismatch prevents Cold learning
  4. LLM overcools: gene

In [18]:
# ── SAVE ALL RESULTS ──────────────────────────────────────────────────────────

# Save generated synthetic Cold rows
synthetic_cold_df.to_csv("llm_synthetic_cold_rows.csv", index=False)

# Save complete results summary
results_summary = {
    "experiment": "LLM Synthetic Data Generation",
    "model": "gpt-4o-mini",
    "n_synthetic_cold_rows": 100,
    "baseline_f1_7class": round(baseline_f1_7, 4),
    "baseline_f1_3class": round(baseline_f1_3, 4),
    "tstr_f1_7class": round(tstr_f1_7, 4),
    "tstr_cold_f1": round(tstr_cold_f1, 4),
    "tstr_vs_baseline": round(tstr_f1_7 - baseline_f1_7, 4),
    "augmented_f1_7class": round(aug_f1_7, 4),
    "augmented_cold_f1": round(aug_cold_f1, 4),
    "augmented_vs_baseline": round(aug_f1_7 - baseline_f1_7, 4),
    "generation_latency_ms_per_row": round(generation_stats["latency_per_row_ms"], 2),
    "rf_inference_latency_ms_per_row": round(tstr_latency, 4),
    "fidelity": {
        "ambient_temp_real_cold_mean": round(train_df[train_df["Label"]==-3]["Ambient_Temperature"].mean(), 2),
        "ambient_temp_synth_cold_mean": round(synthetic_cold_df["Ambient_Temperature"].mean(), 2),
        "wrist_skin_real_cold_mean": round(train_df[train_df["Label"]==-3]["Wrist_Skin_Temperature"].mean(), 2),
        "wrist_skin_synth_cold_mean": round(synthetic_cold_df["Wrist_Skin_Temperature"].mean(), 2),
    },
    "cold_f1_all_conditions": 0.00,
    "key_finding": "LLM overcools — generates Cold at 19C vs real Cold at 22C. Between-subject gap manifests in generation."
}

import json
with open("llm_generation_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)

print("Files saved:")
print("  llm_synthetic_cold_rows.csv")
print("  llm_generation_results.json")
print("\nReady to push to GitHub.")

Files saved:
  llm_synthetic_cold_rows.csv
  llm_generation_results.json

Ready to push to GitHub.


In [19]:
import subprocess
import os
import shutil

os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/repo_gen'])

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # never hardcode a token literal here
GITHUB_USERNAME = "aisha13dikko-sudo"
REPO_NAME = "using-synthetic-data-for-thermal-comfort-classification"

# Clone
clone_result = subprocess.run([
    'git', 'clone',
    f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git',
    '/content/repo_gen'
], capture_output=True, text=True)

print("Clone:", clone_result.stderr.strip())

Clone: Cloning into '/content/repo_gen'...


In [20]:
os.chdir('/content/repo_gen')

subprocess.run(['git', 'add', 'llm_generation/'])

subprocess.run([
    'git', 'commit', '-m',
    'feat: add LLM synthetic Cold generation results (TSTR +0.0174, augmented -0.0130, Cold F1=0.00, latency 2142ms/row)'
])

result = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    capture_output=True, text=True
)

print("Output:", result.stdout)
print("Errors:", result.stderr)

Output: 
Errors: Everything up-to-date



In [21]:
os.chdir('/content/repo_gen')

# Force add even if git thinks nothing changed
subprocess.run(['git', 'add', '-f', 'llm_generation/'])

# Check what is staged
status = subprocess.run(['git', 'status'], capture_output=True, text=True)
print("Status:", status.stdout)

Status: On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean



In [23]:
os.chdir('/content')

# Create folder and copy files
os.makedirs('/content/repo_gen/llm_generation', exist_ok=True)

shutil.copy('/content/llm_synthetic_cold_rows.csv',
            '/content/repo_gen/llm_generation/')
shutil.copy('/content/llm_generation_results.json',
            '/content/repo_gen/llm_generation/')

print("Files copied:")
print(os.listdir('/content/repo_gen/llm_generation/'))

# Now push
os.chdir('/content/repo_gen')
subprocess.run(['git', 'add', 'llm_generation/'])

status = subprocess.run(['git', 'status'], capture_output=True, text=True)
print("\nStatus:", status.stdout)

subprocess.run([
    'git', 'commit', '-m',
    'feat: add LLM synthetic Cold generation results (TSTR +0.0174, augmented -0.0130, Cold F1=0.00)'
])

result = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    capture_output=True, text=True
)
print("Push:", result.stderr)

Files copied:
['llm_generation_results.json', 'llm_synthetic_cold_rows.csv']

Status: On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   llm_generation/llm_generation_results.json
	new file:   llm_generation/llm_synthetic_cold_rows.csv


Push: Everything up-to-date



In [24]:
os.chdir('/content/repo_gen')

result_commit = subprocess.run([
    'git', 'commit', '-m',
    'feat: add LLM synthetic Cold generation results'
], capture_output=True, text=True)

print("Commit:", result_commit.stdout)
print("Commit errors:", result_commit.stderr)

result_push = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    capture_output=True, text=True
)
print("Push:", result_push.stdout)
print("Push errors:", result_push.stderr)

Commit: 
Commit errors: Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@75af840381d3.(none)')

Push: 
Push errors: Everything up-to-date



In [25]:
os.chdir('/content/repo_gen')

# Set identity first
subprocess.run(['git', 'config', 'user.email', 'aisha.dikko.25@ucl.ac.uk'])
subprocess.run(['git', 'config', 'user.name', 'aisha13dikko-sudo'])

# Now commit and push
result_commit = subprocess.run([
    'git', 'commit', '-m',
    'feat: add LLM synthetic Cold generation results'
], capture_output=True, text=True)

print("Commit:", result_commit.stdout)

result_push = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    capture_output=True, text=True
)
print("Push:", result_push.stderr)

Commit: [main 2e78153] feat: add LLM synthetic Cold generation results
 2 files changed, 124 insertions(+)
 create mode 100644 llm_generation/llm_generation_results.json
 create mode 100644 llm_generation/llm_synthetic_cold_rows.csv

Push: To https://github.com/aisha13dikko-sudo/using-synthetic-data-for-thermal-comfort-classification.git
   9ca2289..2e78153  main -> main

